<h1>Silver GTFS<h1>

<h2>Imports<h2>

<h3>Spark and Initialization<h3>

In [ ]:
import findspark
findspark.init()
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("Warsaw_Bus_Project") \
    .config("spark.jars.packages", "org.postgresql:postgresql:42.6.0") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "200") \
    .getOrCreate()

<h3>Imports<h3>

In [ ]:
from datetime import datetime , timedelta
import sys
import gc
sys.path.append('../work')
from config import db_properties,DB_CONFIG,jdbc_url
import os
os.environ["PYARROW_IGNORE_TIMEZONE"] = "1"
import pandas as pd
import numpy as np
import pyspark.pandas as ps
from pyspark.sql.types import FloatType, IntegerType, BooleanType
from pyspark.sql import Row
import psycopg2
from psycopg2 import sql
import pyspark.sql.functions as sf
import zipfile
import urllib.request
from pyspark.testing import assertDataFrameEqual
from pyspark.sql import functions as F
import geopandas as gpd
from shapely.geometry import Point

<h3>Creating Silver schema<h3>

In [3]:
conn = psycopg2.connect(**DB_CONFIG)
cur = conn.cursor()
cur.execute("CREATE SCHEMA IF NOT EXISTS silver;")
conn.commit()
conn.close()

<h2>Silver Stops Table<h2>

<h3>Extracting Bronze Stops<h3>

In [4]:
df_stops_bronze = spark.read.jdbc(url=jdbc_url, table="bronze.stops", properties=db_properties)

<h3>Transforming Stops data<h3>

In [5]:
df_stops_silver = df_stops_bronze.select(
    "stop_id", "stop_code", "stop_name", "stop_lat", "stop_lon"
)\
                .withColumn("stop_id",F.col("stop_id").cast(IntegerType()))\
                .withColumn("stop_lat",F.col("stop_lat").cast(FloatType()))\
                .withColumn("stop_lon",F.col("stop_lon").cast(FloatType()))

<h3>Adding District to Data<h3>

In [6]:
pdf_stops = df_stops_silver.select("stop_id", "stop_lat", "stop_lon").toPandas()
geometry = [Point(xy) for xy in zip(pdf_stops.stop_lon, pdf_stops.stop_lat)]
gdf_stops = gpd.GeoDataFrame(pdf_stops, geometry=geometry, crs="EPSG:4326")
gdf_districts = gpd.read_file("../Silver/warszawa-dzielnice.geojson")
gdf_stops_with_districts = gpd.sjoin(gdf_stops, gdf_districts, how="left", predicate="within")
gdf_stops_with_districts['name'] = gdf_stops_with_districts['name'].fillna("Poza Warszawa")

In [7]:
df_districts = spark.createDataFrame(gdf_stops_with_districts[["stop_id", "name"]].rename(columns={"name": "district"}))
df_stops_silver = df_stops_silver.join(df_districts, on="stop_id", how="left")
df_stops_silver = df_stops_silver.where(F.col("district")!="Warszawa")

<h3>Writing to DB<h3>

In [8]:
df_stops_silver.write.jdbc(url=jdbc_url, table="silver.stops", mode="overwrite", properties=db_properties)

<h2>Silver Routes Table<h2>

<h3>Extracting Bronze Routes<h3>

In [9]:
df_routes_bronze = spark.read.jdbc(url=jdbc_url, table="bronze.routes", properties=db_properties)
df_routes_bronze.show(5)

+--------+---------+----------------+--------------------+----------+-----------+----------------+
|route_id|agency_id|route_short_name|     route_long_name|route_type|route_color|route_text_color|
+--------+---------+----------------+--------------------+----------+-----------+----------------+
|       1|        0|               1|Rondo Daszyńskieg...|         0|     B60000|          FFFFFF|
|      10|        0|              10|Os. Górczewska – ...|         0|     B60000|          FFFFFF|
|     102|        0|             102|Metro Stadion Nar...|         3|     880077|          FFFFFF|
|     103|        0|             103|Metro Młociny – D...|         3|     880077|          FFFFFF|
|     104|        0|             104|Metro Bródno – Br...|         3|     880077|          FFFFFF|
+--------+---------+----------------+--------------------+----------+-----------+----------------+
only showing top 5 rows



<h3>Transforming Routes data<h3>

In [10]:
df_routes_bronze = df_routes_bronze.drop("route_color","route_text_color")

In [ ]:
df_routes_silver = df_routes_bronze\
                .withColumn("agency_id",F.col("agency_id").cast(IntegerType()))\
                .withColumn("route_type",F.col("route_type").cast(IntegerType()))\
                .withColumnRenamed("route_short_name", "route_name")\
                .withColumnRenamed("route_long_name", "route_desc")
df_routes_silver = df_routes_silver.where(F.col("route_type")==3)

DataFrame[route_id: string, agency_id: int, route_name: string, route_desc: string, route_type: int]

<h3>Writing to DB<h3>

In [12]:
df_routes_silver.write.jdbc(url=jdbc_url, table="silver.routes", mode="overwrite", properties=db_properties)

<h2>Silver Trips Table<h2>

<h3>Extracting Bronze Trips<h3>

In [13]:
df_trips_bronze = spark.read.jdbc(url=jdbc_url, table="bronze.trips", properties=db_properties)
df_trips_bronze.show(5)

+--------------------+--------+--------------+--------+---------------+--------------+------------+-----------+---------------------+-----------------+----------------+------------+----------+
|             trip_id|route_id|    service_id|shape_id|trip_short_name| trip_headsign|direction_id|exceptional|wheelchair_accessible|         block_id|block_short_name|variant_code|fleet_type|
+--------------------+--------+--------------+--------+---------------+--------------+------------+-----------+---------------------+-----------------+----------------+------------+----------+
|2026-04-24:4:PtS:...|       4|2026-04-24:PtS|   0:185|           NULL|       Wyścigi|           0|          1|                    1|2026-04-24:663200|              13|     TD-5WYS|      120N|
|2026-04-24:4:PtS:...|       4|2026-04-24:PtS|   0:182|           NULL|Żerań Wschodni|           1|          0|                    1|2026-04-24:663200|              13|      TP-ZEW|      120N|
|2026-04-24:4:PtS:...|       4|2026

<h3>Transforming Stops data<h3>

In [14]:
df_trips_bronze = df_trips_bronze.drop(
    "shape_id", "trip_short_name", "exceptional", "wheelchair_accessible",
    "block_id","block_short_name", "variant_code", "fleet_type"
)

In [15]:
df_trips_silver = df_trips_bronze\
                .withColumn("direction_id",F.col("direction_id").cast(IntegerType())) \

df_trips_silver

DataFrame[trip_id: string, route_id: string, service_id: string, trip_headsign: string, direction_id: int]

<h3>Writing to DB<h3>

In [16]:
df_trips_silver.write.jdbc(url=jdbc_url, table="silver.trips", mode="overwrite", properties=db_properties)

<h2>Silver Stop_Times Table<h2>

<h3>Extracting Bronze Stop_Times<h3>

In [21]:
df_stop_times_bronze = spark.read.jdbc(url=jdbc_url, table="bronze.stop_times", properties=db_properties)
df_stop_times_bronze

DataFrame[trip_id: string, stop_sequence: string, stop_id: string, arrival_time: string, departure_time: string, pickup_type: string, drop_off_type: string, shape_dist_traveled: string]

<h3>Transforming Stop_Times data</h3>

In [22]:
df_stop_times_bronze = df_stop_times_bronze\
                .withColumn("stop_sequence",F.col("stop_sequence").cast(IntegerType()))\
                .withColumn("pickup_type",F.col("pickup_type").cast(IntegerType()))\
                .withColumn("drop_off_type",F.col("drop_off_type").cast(IntegerType()))\
                .withColumn("stop_id",F.col("stop_id").cast(IntegerType()))\
                .withColumn("shape_dist_traveled",F.col("shape_dist_traveled").cast(FloatType()))
df_stop_times_bronze

DataFrame[trip_id: string, stop_sequence: int, stop_id: int, arrival_time: string, departure_time: string, pickup_type: int, drop_off_type: int, shape_dist_traveled: float]

In [23]:
df_stop_times_bronze.show(5)

+--------------------+-------------+-------+------------+--------------+-----------+-------------+-------------------+
|             trip_id|stop_sequence|stop_id|arrival_time|departure_time|pickup_type|drop_off_type|shape_dist_traveled|
+--------------------+-------------+-------+------------+--------------+-----------+-------------+-------------------+
|2026-04-16:102:Pc...|            0|   NULL|    14:03:00|      14:03:00|          1|            1|                0.0|
|2026-04-16:102:Pc...|            1| 210804|    14:20:00|      14:20:00|          1|            1|           7.647715|
|2026-04-16:102:Pc...|            0| 210804|    14:26:00|      14:26:00|          0|            0|                0.0|
|2026-04-16:102:Pc...|            1| 210903|    14:27:00|      14:27:00|          0|            0|           0.340602|
|2026-04-16:102:Pc...|            2| 211001|    14:29:00|      14:29:00|          0|            0|            0.91718|
+--------------------+-------------+-------+----

<h3>Setting up timestamps<h3>

In [24]:
df_stop_times_joined = df_stop_times_bronze.join(df_trips_silver, on="trip_id", how="inner")\
                

In [25]:
df_stop_times_joined.show(5)

+--------------------+-------------+-------+------------+--------------+-----------+-------------+-------------------+--------+--------------+--------------------+------------+
|             trip_id|stop_sequence|stop_id|arrival_time|departure_time|pickup_type|drop_off_type|shape_dist_traveled|route_id|    service_id|       trip_headsign|direction_id|
+--------------------+-------------+-------+------------+--------------+-----------+-------------+-------------------+--------+--------------+--------------------+------------+
|2026-04-16:102:Pc...|            0| 123107|    05:45:00|      05:45:00|          0|            0|                0.0|     102|2026-04-16:PcS|PKP Olszynka Groc...|           0|
|2026-04-16:102:Pc...|            1| 123204|    05:46:00|      05:46:00|          3|            3|           0.622683|     102|2026-04-16:PcS|PKP Olszynka Groc...|           0|
|2026-04-16:102:Pc...|            2| 123102|    05:47:00|      05:47:00|          0|            0|           1.0391

In [26]:
df_stop_times_silver = df_stop_times_joined \
    .withColumn(
        "op_date",
        F.coalesce(
            F.to_date(F.regexp_extract(F.col("service_id"), r"^(\\d{4}-\\d{2}-\\d{2})", 1), "yyyy-MM-dd"),
            F.current_date()
        )
    ) \
    .withColumn("arrival_sec", F.col("arrival_time").substr(1, 2).cast(IntegerType()) * 3600 + \
                                F.col("arrival_time").substr(4, 2).cast(IntegerType()) * 60 + \
                                F.col("arrival_time").substr(7, 2).cast(IntegerType())) \
    .withColumn("departure_sec", F.col("departure_time").substr(1, 2).cast(IntegerType()) * 3600 + \
                                F.col("departure_time").substr(4, 2).cast(IntegerType()) * 60 + \
                                F.col("departure_time").substr(7, 2).cast(IntegerType())) \
    .withColumn("parsed_arrival_date",
                F.expr("cast(to_timestamp(op_date) + make_interval(0, 0, 0, 0, 0, 0, arrival_sec) as timestamp)")) \
    .withColumn("parsed_departure_date",
                F.expr("cast(to_timestamp(op_date) + make_interval(0, 0, 0, 0, 0, 0, departure_sec) as timestamp)"))

In [27]:
df_stop_times_silver = df_stop_times_silver.select(
    "trip_id", "stop_id", "stop_sequence", "pickup_type", 
    "drop_off_type", "shape_dist_traveled",
    F.col("parsed_arrival_date").alias("arrival_time"), F.col("parsed_departure_date").alias("departure_time")
)
df_stop_times_silver

DataFrame[trip_id: string, stop_id: int, stop_sequence: int, pickup_type: int, drop_off_type: int, shape_dist_traveled: float, arrival_time: timestamp, departure_time: timestamp]

<h3>Writing to DB<h3>

In [28]:
df_stop_times_silver.repartition(16).write.jdbc(
    url=jdbc_url,
    table="silver.stop_times",
    mode="overwrite",
    properties=db_properties
)

In [29]:
spark.catalog.clearCache()
gc.collect()
spark.stop()